# BGE-M3 ColBERT 재정렬 평가

로컬 PostgreSQL Hybrid Retrieval이 생성한 후보를 Colab GPU에서 BGE-M3 multi-vector 점수만 사용해 재정렬합니다. 이 노트북은 LLM 근거 유효성 판정을 실행하지 않으며, ColBERT 재정렬의 정확도와 지연시간만 측정합니다.

먼저 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → GPU**를 선택하세요. A100을 권장하며 T4에서도 실행할 수 있습니다.

In [ ]:
%pip install -q FlagEmbedding==1.4.2

In [ ]:
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)
if DEVICE != 'cuda':
    raise RuntimeError('GPU 런타임을 선택한 뒤 다시 실행하세요.')
print('gpu:', torch.cuda.get_device_name(0))

In [ ]:
import json
from google.colab import files

uploaded = files.upload()
input_name = next(name for name in uploaded if name.endswith('.json'))
payload = json.loads(uploaded[input_name].decode('utf-8'))
assert payload['schema_version'] == 1
print('cases:', len(payload['cases']))
print('candidates:', sum(len(case['candidates']) for case in payload['cases']))

In [ ]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel(payload['reranker_model'], devices=DEVICE)
print('loaded:', payload['reranker_model'])

In [ ]:
from time import perf_counter

case_results = []
for case in payload['cases']:
    candidates = case['candidates']
    started = perf_counter()
    if candidates:
        pairs = [(case['question'], candidate['content']) for candidate in candidates]
        score_output = model.compute_score(
            pairs,
            batch_size=2,
            max_query_length=payload['max_query_length'],
            max_passage_length=payload['max_passage_length'],
            weights_for_different_modes=[0.0, 0.0, 1.0],
        )
        scores = score_output['colbert']
        if isinstance(scores, (int, float)):
            scores = [scores]
        ranked = [dict(candidate, colbert_score=float(score)) for candidate, score in zip(candidates, scores)]
        ranked.sort(key=lambda row: (-row['colbert_score'], -row['rrf_score'], row['authority_tier'], row['parent_chunk_id']))
    else:
        ranked = []
    latency_ms = (perf_counter() - started) * 1000
    source_order = list(dict.fromkeys(row['source_id'] for row in ranked))
    expected = set(case['expected_source_ids'])
    hit_rank = next((index for index, source_id in enumerate(source_order, 1) if source_id in expected), None)
    case_results.append({
        'case_id': case['case_id'],
        'category': case['category'],
        'question': case['question'],
        'expected_status': case['expected_status'],
        'expected_source_ids': case['expected_source_ids'],
        'source_order': source_order,
        'source_recall_at_k': (len(expected.intersection(source_order)) / len(expected)) if expected else None,
        'reciprocal_rank': (1.0 / hit_rank) if hit_rank else (0.0 if expected else None),
        'latency_ms': latency_ms,
        'top_candidates': ranked[:5],
    })
    print(case['case_id'], f'{latency_ms:.1f}ms', source_order[:3])

In [ ]:
from statistics import mean, median
from google.colab import files

source_cases = [result for result in case_results if result['source_recall_at_k'] is not None]
latencies = sorted(result['latency_ms'] for result in case_results)
p95_index = max(0, min(len(latencies) - 1, int(0.95 * len(latencies) + 0.999) - 1))
summary = {
    'case_count': len(case_results),
    'source_case_count': len(source_cases),
    'recall_at_k': mean(result['source_recall_at_k'] for result in source_cases),
    'mrr': mean(result['reciprocal_rank'] for result in source_cases),
    'latency_mean_ms': mean(latencies),
    'latency_p50_ms': median(latencies),
    'latency_p95_ms': latencies[p95_index],
    'note': 'no_evidence는 ColBERT 점수만으로 판정하지 않으며 full LLM gate 평가에서 측정한다.',
}
report = {'summary': summary, 'cases': case_results}
output_name = 'colab_bge_m3_rerank_results.json'
with open(output_name, 'w', encoding='utf-8') as file:
    json.dump(report, file, ensure_ascii=False, indent=2)
print(json.dumps(summary, ensure_ascii=False, indent=2))
files.download(output_name)